# Weekday Profile (2023 preds) → 2026 Full-Year Predictions (Profile-based)

This notebook does **two things** end-to-end:

1. Build **weekday profiles** from existing prediction files (e.g., `preds/final/year=2023/...` or `preds/mvstgcn/year=2023/...`).
   - Profile = average probability per `(sourceelementkey, dow, slot)` where:
     - `dow` = local day-of-week (Mon=0 … Sun=6)
     - `slot` = 15-min index in day (0…95)

2. Use that profile to generate **2026 predictions for every 15-min timestamp of every week** (profile lookup).
   - Optionally override **Sunday + free days** to use **Saturday (dow=5)** pattern (same spirit as our existing fallback).



In [1]:

# =========================
# Config
# =========================

S3_ROOT = "s3://smart-park-seattle/parking_v2/" 

# Where to read 2023 predictions from
YEAR_REF = 2023

PRED_SOURCE_MODEL = "final"

# Targets to build profiles/preds for
TARGETS = ["y_15", "y_30"]

# Output year
YEAR_OUT = 2026

# Timezone for dow/slot computations (must match project convention)
TZ_LOCAL = "America/Los_Angeles"

# If True: for generated 2026 data, timestamps that are Sunday OR free-day
# will use Saturday profile (dow forced to 5).
OVERRIDE_SUN_AND_FREE_TO_SAT = True

# Free days CSV location:
# - relative to S3_ROOT prefix, usually "meta/free_days.csv"
# - or full s3://... path
FREE_DAYS_KEY = "meta/free_days.csv"

# Add noise
UNCERTAINTY_LOW = 0.05
UNCERTAINTY_HIGH = 0.10

# Output location under preds/<OUT_MODEL_NAME>/year=YEAR_OUT/target=.../week=.../pred.csv.gz
OUT_MODEL_NAME = "profile_year"

# Behavior
OVERWRITE = False

# How many weeks to run as a smoke test before full year:
SMOKE_TEST_WEEKS = 1


In [2]:

import io, os, re, gzip, math, json
from dataclasses import dataclass
from typing import List, Set, Tuple, Literal

import boto3
import numpy as np
import pandas as pd

# We try to use your project's parse_s3_uri if available; otherwise use a small fallback.
try:
    from parking_processing.utils.s3 import parse_s3_uri
except Exception:
    def parse_s3_uri(uri: str) -> Tuple[str, str]:
        if not uri.startswith("s3://"):
            raise ValueError(f"Expected s3://..., got {uri!r}")
        rest = uri[5:]
        parts = rest.split("/", 1)
        bucket = parts[0]
        prefix = parts[1] if len(parts) == 2 else ""
        prefix = prefix.lstrip("/")
        if prefix and not prefix.endswith("/"):
            prefix += "/"
        return bucket, prefix

print("imports ok")


imports ok


In [3]:

# -------------------------
# S3 helpers
# -------------------------

s3 = boto3.client("s3")
BUCKET, PREFIX = parse_s3_uri(S3_ROOT)

def _parse_s3_object_uri(uri: str) -> Tuple[str, str]:
    if not isinstance(uri, str) or not uri.startswith("s3://"):
        raise ValueError(f"Expected s3://... uri, got: {uri!r}")
    rest = uri[5:]
    parts = rest.split("/", 1)
    b = parts[0]
    k = parts[1] if len(parts) == 2 else ""
    return b, k.lstrip("/")

def s3_read_csv_gz(bucket: str, key: str) -> pd.DataFrame:
    obj = s3.get_object(Bucket=bucket, Key=key)
    raw = obj["Body"].read()
    with gzip.GzipFile(fileobj=io.BytesIO(raw), mode="rb") as f:
        return pd.read_csv(f)

def s3_read_csv(bucket: str, key: str) -> pd.DataFrame:
    obj = s3.get_object(Bucket=bucket, Key=key)
    raw = obj["Body"].read()
    return pd.read_csv(io.BytesIO(raw))

def s3_put_csv_gz(bucket: str, key: str, df: pd.DataFrame):
    bio = io.BytesIO()
    with gzip.GzipFile(fileobj=bio, mode="wb") as gz:
        df.to_csv(io.TextIOWrapper(gz, encoding="utf-8", newline=""), index=False)
    bio.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=bio.getvalue())

def s3_upload_file(local_path: str, bucket: str, key: str):
    s3.upload_file(local_path, bucket, key)

def s3_put_success(bucket: str, key: str, msg: str = "ok\n"):
    s3.put_object(Bucket=bucket, Key=key, Body=msg.encode("utf-8"))

def s3_exists(bucket: str, key: str) -> bool:
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except Exception:
        return False

def list_pred_keys(year: int, target: str, model_name: str) -> List[str]:
    # model_name: "final" or "mvstgcn" or others
    pfx = f"{PREFIX}preds/{model_name}/year={year}/target={target}/"
    keys=[]
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET, Prefix=pfx):
        for it in page.get("Contents", []):
            k = it["Key"]
            if k.endswith("/pred.csv.gz"):
                keys.append(k)
    return sorted(keys)

def week_grid_utc(week_start: str, tz_local: str) -> pd.DatetimeIndex:
    start_local = pd.Timestamp(f"{week_start} 00:00:00", tz=tz_local)
    start_utc = start_local.tz_convert("UTC")
    return pd.date_range(start=start_utc, periods=7*96, freq="15min", tz="UTC")

def monday_weeks_for_year(year: int) -> List[str]:
    d = pd.Timestamp(f"{year}-01-01")
    while d.weekday() != 0:
        d += pd.Timedelta(days=1)
    weeks=[]
    while d.year == year:
        weeks.append(d.strftime("%Y-%m-%d"))
        d += pd.Timedelta(days=7)
    return weeks

def load_free_days(free_days_key: str) -> Set[pd.Timestamp]:
    # returns set of date-only Timestamp (local date)
    if free_days_key.startswith("s3://"):
        b, k = _parse_s3_object_uri(free_days_key)
        df = s3_read_csv(b, k)
    else:
        df = s3_read_csv(BUCKET, f"{PREFIX}{free_days_key}".rstrip("/"))
    df.columns = [c.lower() for c in df.columns]
    if "is_free" in df.columns:
        s = df["is_free"]
        if s.dtype == bool:
            mask = s
        else:
            mask = s.astype(str).str.lower().isin(["1","true","t","yes","y"])
        df = df[mask].copy()
    # date column guessing
    for col in ["date_local","observed_date","date","day"]:
        if col in df.columns:
            dcol = col
            break
    else:
        raise KeyError(f"free_days.csv missing date column. columns={list(df.columns)}")
    s = pd.to_datetime(df[dcol], errors="coerce").dropna()
    return set(pd.Timestamp(x.date()) for x in s)

print("helpers ready:", BUCKET, PREFIX)


helpers ready: smart-park-seattle parking_v2/


In [4]:

# -------------------------
# Build weekday profile from 2023 predictions
# -------------------------

def build_weekday_profile_from_preds(year_ref: int, target: str, model_name: str, tz_local: str) -> pd.DataFrame:
    prob_col = "p15" if target == "y_15" else "p30"
    keys = list_pred_keys(year_ref, target, model_name)
    if not keys:
        raise FileNotFoundError(f"No pred.csv.gz found for year={year_ref}, target={target}, model={model_name}")

    parts=[]
    for k in keys:
        df = s3_read_csv_gz(BUCKET, k)
        if df.empty:
            continue
        need = {"ts15_utc","sourceelementkey",prob_col}
        if not need.issubset(df.columns):
            continue

        ts_local = pd.to_datetime(df["ts15_utc"], utc=True).dt.tz_convert(tz_local)
        dow = ts_local.dt.dayofweek.astype(int)  # Mon=0..Sun=6
        slot = (ts_local.dt.hour * 4 + (ts_local.dt.minute // 15)).astype(int)

        tmp = df[["sourceelementkey", prob_col]].copy()
        tmp["dow"] = dow
        tmp["slot"] = slot
        tmp.rename(columns={prob_col: "p"}, inplace=True)

        parts.append(tmp[["sourceelementkey","dow","slot","p"]])

    if not parts:
        raise RuntimeError("No usable rows found to build profile.")

    all_df = pd.concat(parts, ignore_index=True)
    prof = (all_df
            .groupby(["sourceelementkey","dow","slot"], as_index=False)["p"]
            .mean()
            .rename(columns={"p":"p_mean"}))

    prof["sourceelementkey"] = prof["sourceelementkey"].astype(int)
    prof["dow"] = prof["dow"].astype(int)
    prof["slot"] = prof["slot"].astype(int)
    prof["p_mean"] = pd.to_numeric(prof["p_mean"], errors="coerce").fillna(0.5).clip(0.0, 1.0)
    return prof

profiles = {}
for t in TARGETS:
    print("building profile for", t, "from", PRED_SOURCE_MODEL, "year", YEAR_REF)
    prof = build_weekday_profile_from_preds(YEAR_REF, t, PRED_SOURCE_MODEL, TZ_LOCAL)
    profiles[t] = prof
    print("profile rows:", len(prof), "unique nodes:", prof["sourceelementkey"].nunique())


building profile for y_15 from final year 2023
profile rows: 1005480 unique nodes: 1512
building profile for y_30 from final year 2023
profile rows: 1005480 unique nodes: 1512


In [8]:
def fill_missing_dow_slot(prof: pd.DataFrame) -> pd.DataFrame:
    # prof columns: sourceelementkey, dow, slot, p_mean
    slot_mean = (prof.groupby(["sourceelementkey","slot"], as_index=False)["p_mean"]
                    .mean()
                    .rename(columns={"p_mean":"p_slot"}))

    # (node, dow, slot) grid 
    nodes = prof["sourceelementkey"].dropna().astype(int).unique()
    full = (
        pd.MultiIndex.from_product([nodes, range(7), range(96)],
                                   names=["sourceelementkey","dow","slot"])
        .to_frame(index=False)
    )

    out = full.merge(prof, on=["sourceelementkey","dow","slot"], how="left")
    out = out.merge(slot_mean, on=["sourceelementkey","slot"], how="left")

    # 1) missing p_mean -> p_slot
    out["p_mean"] = out["p_mean"].fillna(out["p_slot"])

    # 2) if still missing -> global mean
    out["p_mean"] = out["p_mean"].fillna(float(prof["p_mean"].mean()) if len(prof) else 0.5)

    out = out.drop(columns=["p_slot"])
    out["p_mean"] = out["p_mean"].clip(0.0, 1.0)
    return out

# Apply
for t in TARGETS:
    profiles[t] = fill_missing_dow_slot(profiles[t])
    prof = profiles[t]
    cov = prof[["dow","slot"]].drop_duplicates().shape[0]
    print(t, "coverage:", cov, "/ expected", 7*96)

y_15 coverage: 672 / expected 672
y_30 coverage: 672 / expected 672


In [9]:

# -------------------------
# Save weekday profiles to S3 (optional but recommended)
# -------------------------

profile_uris = {}

for t, prof in profiles.items():
    out_key = f"{PREFIX}meta/weekday_profile/year_ref={YEAR_REF}/target={t}/profile.csv.gz"
    s3_put_csv_gz(BUCKET, out_key, prof)
    uri = f"s3://{BUCKET}/{out_key}"
    profile_uris[t] = uri
    print("saved:", uri)


saved: s3://smart-park-seattle/parking_v2/meta/weekday_profile/year_ref=2023/target=y_15/profile.csv.gz
saved: s3://smart-park-seattle/parking_v2/meta/weekday_profile/year_ref=2023/target=y_30/profile.csv.gz


In [10]:

# -------------------------
# Generate 2026 preds by profile lookup
# -------------------------

def make_profile_preds_for_week(
    week: str,
    target: str,
    prof: pd.DataFrame,
    tz_local: str,
    override_sun_and_free_to_sat: bool,
    free_days: Set[pd.Timestamp],
    uncertainty_low: float,
    uncertainty_high: float,
) -> pd.DataFrame:
    ts_utc = week_grid_utc(week, tz_local)
    ts_local = ts_utc.tz_convert(tz_local)

    dow = ts_local.dayofweek.astype(int)                # Mon=0..Sun=6
    slot = (ts_local.hour * 4 + (ts_local.minute // 15)).astype(int)

    if override_sun_and_free_to_sat:
        free_set = set(d.date() for d in free_days)
        local_dates = ts_local.date
        is_sun_or_free = np.array([(d.weekday() == 6) or (d in free_set) for d in local_dates], dtype=bool)
        dow = np.where(is_sun_or_free, 5, dow)

    base = pd.DataFrame({"ts15_utc": ts_utc, "dow": dow, "slot": slot})

    # broadcast join: (ts,dow,slot) x (node,dow,slot) -> (ts,node)
    df = base.merge(prof, on=["dow","slot"], how="left")

    # fill missing p_mean
    node_mean = prof.groupby("sourceelementkey", as_index=False)["p_mean"].mean().rename(columns={"p_mean":"p_node"})
    df = df.merge(node_mean, on="sourceelementkey", how="left")
    df["p_mean"] = df["p_mean"].fillna(df["p_node"])
    df["p_mean"] = df["p_mean"].fillna(float(prof["p_mean"].mean()) if len(prof) else 0.5)

    p = df["p_mean"].to_numpy(dtype=float)

    # optional uncertainty
    if uncertainty_high and uncertainty_high > 0:
        rng = np.random.default_rng(abs(hash((week, target))) % (2**32))
        u_hi = rng.uniform(1.0 - uncertainty_high, 1.0 + uncertainty_high, size=len(p))
        if uncertainty_low != uncertainty_high:
            u_lo = rng.uniform(1.0 - uncertainty_low, 1.0 + uncertainty_low, size=len(p))
            mix = rng.uniform(0, 1, size=len(p))
            u = mix * u_hi + (1 - mix) * u_lo
        else:
            u = u_hi
        p = (p * u).clip(0.0, 1.0)

    out_col = "p15" if target == "y_15" else "p30"
    out = df[["ts15_utc","sourceelementkey"]].copy()
    out[out_col] = p
    out["ts15_utc"] = pd.to_datetime(out["ts15_utc"], utc=True)
    out["sourceelementkey"] = out["sourceelementkey"].astype(int)
    out = out.sort_values(["ts15_utc","sourceelementkey"]).reset_index(drop=True)
    return out

def write_profile_preds_week_to_s3(
    year_out: int,
    week: str,
    target: str,
    df: pd.DataFrame,
    out_model_name: str,
    overwrite: bool,
):
    pred_key = f"{PREFIX}preds/{out_model_name}/year={year_out}/target={target}/week={week}/pred.csv.gz"
    suc_key  = f"{PREFIX}preds/{out_model_name}/year={year_out}/target={target}/week={week}/_SUCCESS"

    if (not overwrite) and s3_exists(BUCKET, suc_key):
        print("[skip]", week, target, "already success")
        return

    os.makedirs("/tmp/profile_year", exist_ok=True)
    local = f"/tmp/profile_year/pred_{year_out}_{week}_{target}.csv.gz"
    with gzip.open(local, "wt", encoding="utf-8") as f:
        df.to_csv(f, index=False)

    s3_upload_file(local, BUCKET, pred_key)
    s3_put_success(BUCKET, suc_key, "ok\n")
    print("[ok]", week, target, "->", f"s3://{BUCKET}/{pred_key}")

# load free days if needed
free_days = set()
if OVERRIDE_SUN_AND_FREE_TO_SAT:
    free_days = load_free_days(FREE_DAYS_KEY)
    print("free_days loaded:", len(free_days))

weeks_out = ["2025-12-29"] + monday_weeks_for_year(YEAR_OUT)
print("weeks_out:", len(weeks_out), weeks_out[0], weeks_out[-1])

# smoke test
smoke = weeks_out[:max(1, SMOKE_TEST_WEEKS)]
print("SMOKE:", smoke)

for t in TARGETS:
    prof = profiles[t]
    for wk in smoke:
        df = make_profile_preds_for_week(
            week=wk,
            target=t,
            prof=prof,
            tz_local=TZ_LOCAL,
            override_sun_and_free_to_sat=OVERRIDE_SUN_AND_FREE_TO_SAT,
            free_days=free_days,
            uncertainty_low=UNCERTAINTY_LOW,
            uncertainty_high=UNCERTAINTY_HIGH,
        )
        write_profile_preds_week_to_s3(YEAR_OUT, wk, t, df, OUT_MODEL_NAME, OVERWRITE)


free_days loaded: 632
weeks_out: 52 2026-01-05 2026-12-28
SMOKE: ['2026-01-05']
[ok] 2026-01-05 y_15 -> s3://smart-park-seattle/parking_v2/preds/profile_year/year=2026/target=y_15/week=2026-01-05/pred.csv.gz
[ok] 2026-01-05 y_30 -> s3://smart-park-seattle/parking_v2/preds/profile_year/year=2026/target=y_30/week=2026-01-05/pred.csv.gz


In [11]:

# -------------------------
# smoke test looks good! now Full-year run 
# -------------------------

for t in TARGETS:
    prof = profiles[t]
    for wk in weeks_out:
        df = make_profile_preds_for_week(
            week=wk,
            target=t,
            prof=prof,
            tz_local=TZ_LOCAL,
            override_sun_and_free_to_sat=OVERRIDE_SUN_AND_FREE_TO_SAT,
            free_days=free_days,
            uncertainty_low=UNCERTAINTY_LOW,
            uncertainty_high=UNCERTAINTY_HIGH,
        )
        write_profile_preds_week_to_s3(YEAR_OUT, wk, t, df, OUT_MODEL_NAME, OVERWRITE)


[skip] 2026-01-05 y_15 already success
[ok] 2026-01-12 y_15 -> s3://smart-park-seattle/parking_v2/preds/profile_year/year=2026/target=y_15/week=2026-01-12/pred.csv.gz
[ok] 2026-01-19 y_15 -> s3://smart-park-seattle/parking_v2/preds/profile_year/year=2026/target=y_15/week=2026-01-19/pred.csv.gz
[ok] 2026-01-26 y_15 -> s3://smart-park-seattle/parking_v2/preds/profile_year/year=2026/target=y_15/week=2026-01-26/pred.csv.gz
[ok] 2026-02-02 y_15 -> s3://smart-park-seattle/parking_v2/preds/profile_year/year=2026/target=y_15/week=2026-02-02/pred.csv.gz
[ok] 2026-02-09 y_15 -> s3://smart-park-seattle/parking_v2/preds/profile_year/year=2026/target=y_15/week=2026-02-09/pred.csv.gz
[ok] 2026-02-16 y_15 -> s3://smart-park-seattle/parking_v2/preds/profile_year/year=2026/target=y_15/week=2026-02-16/pred.csv.gz
[ok] 2026-02-23 y_15 -> s3://smart-park-seattle/parking_v2/preds/profile_year/year=2026/target=y_15/week=2026-02-23/pred.csv.gz
[ok] 2026-03-02 y_15 -> s3://smart-park-seattle/parking_v2/preds/

In [12]:

# -------------------------
# Sanity check: read back one generated week
# -------------------------

def read_generated_week(year_out: int, week: str, target: str, out_model_name: str) -> pd.DataFrame:
    key = f"{PREFIX}preds/{out_model_name}/year={year_out}/target={target}/week={week}/pred.csv.gz"
    return s3_read_csv_gz(BUCKET, key)

wk = weeks_out[0]
for t in TARGETS:
    df = read_generated_week(YEAR_OUT, wk, t, OUT_MODEL_NAME)
    col = "p15" if t == "y_15" else "p30"
    print("\n---", t, "week", wk, "---")
    print("shape:", df.shape, "cols:", list(df.columns))
    print("dups(ts,node):", int(df.duplicated(["ts15_utc","sourceelementkey"]).sum()))
    print("p range:", float(df[col].min()), float(df[col].max()))
    print(df.head(3))



--- y_15 week 2026-01-05 ---
shape: (1016064, 3) cols: ['ts15_utc', 'sourceelementkey', 'p15']
dups(ts,node): 0
p range: 0.0002012445519945 1.0
                    ts15_utc  sourceelementkey       p15
0  2026-01-05 08:00:00+00:00              1001  0.707685
1  2026-01-05 08:00:00+00:00              1002  0.730906
2  2026-01-05 08:00:00+00:00              1005  0.770488

--- y_30 week 2026-01-05 ---
shape: (1016064, 3) cols: ['ts15_utc', 'sourceelementkey', 'p30']
dups(ts,node): 0
p range: 0.0007118924789802 1.0
                    ts15_utc  sourceelementkey       p30
0  2026-01-05 08:00:00+00:00              1001  0.630841
1  2026-01-05 08:00:00+00:00              1002  0.657841
2  2026-01-05 08:00:00+00:00              1005  0.694712
